# Create Web Search UC Functions (Free Tier)

This notebook creates Unity Catalog functions that give your Supervisor Agent real-time web search and content extraction capabilities.

## Functions Created:
1. **you_web_search** -- Web and news search via You.com MCP server (free tier, no API key)
2. **you_content_extract** -- Extract page content from any URL (direct HTTP fetch, no API key)
3. **you_research** -- Multi-query web research combining search + content extraction

## Why UC Functions?
- **Free** -- no API key, no signup, no billing
- No Unity Catalog connection or secret scopes needed
- Native integration with the Supervisor Agent

## Prerequisites
- USAGE + CREATE permissions on your catalog.schema
- That's it! No API key needed

In [ ]:
%pip install databricks-sdk unitycatalog-ai typing_extensions>=4.7.0 databricks-connect>=15.1.0 --upgrade
dbutils.library.restartPython()  # Comment out for local execution via dbconnect

## Step 1: Configure Your Catalog and Schema

In [ ]:
# Import configuration
%run ../config.py

# Verify you have permissions
spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE SCHEMA {schema}")
print(f"\u2705 Using {catalog}.{schema}")

## Step 2: Define the UC Functions

Three functions:
1. **you_web_search** -- Calls the You.com MCP endpoint (`https://api.you.com/mcp`) for web search (free tier)
2. **you_content_extract** -- Fetches a URL directly via HTTP and extracts readable text (no external API needed)
3. **you_research** -- Performs multiple web searches and combines results into a research summary

In [ ]:
def _call_you_mcp_tool(tool_name: str, arguments: dict) -> dict:
    """
    Helper: call a tool on the You.com MCP server (free tier, no API key).
    Uses the MCP Streamable HTTP transport with JSON-RPC protocol.
    """
    import json
    import http.client
    import uuid

    MCP_HOST = "api.you.com"
    MCP_PATH = "/mcp"

    conn = http.client.HTTPSConnection(MCP_HOST, timeout=60)

    # Step 1: Initialize MCP session
    init_payload = json.dumps({
        "jsonrpc": "2.0",
        "method": "initialize",
        "params": {
            "protocolVersion": "2024-11-05",
            "capabilities": {},
            "clientInfo": {"name": "databricks-uc-function", "version": "1.0"}
        },
        "id": 1
    })

    headers = {
        "Content-Type": "application/json",
        "Accept": "application/json, text/event-stream"
    }

    conn.request("POST", MCP_PATH, body=init_payload, headers=headers)
    init_resp = conn.getresponse()
    init_body = init_resp.read().decode("utf-8")

    # Capture session ID from response headers if provided
    session_id = init_resp.getheader("Mcp-Session-Id") or init_resp.getheader("mcp-session-id")

    # Parse SSE or JSON response
    def parse_mcp_response(body, content_type):
        if "text/event-stream" in (content_type or ""):
            # Parse SSE: look for data lines containing JSON-RPC responses
            for line in body.split("\n"):
                if line.startswith("data: "):
                    try:
                        msg = json.loads(line[6:])
                        if "result" in msg or "error" in msg:
                            return msg
                    except json.JSONDecodeError:
                        continue
            return {"error": {"message": "No valid JSON-RPC response in SSE stream"}}
        else:
            return json.loads(body)

    init_ct = init_resp.getheader("Content-Type", "")
    init_result = parse_mcp_response(init_body, init_ct)

    if "error" in init_result:
        return {"error": f"MCP init failed: {init_result['error']}"}

    # Step 2: Send initialized notification
    notif_payload = json.dumps({
        "jsonrpc": "2.0",
        "method": "notifications/initialized"
    })
    notif_headers = dict(headers)
    if session_id:
        notif_headers["Mcp-Session-Id"] = session_id

    conn.request("POST", MCP_PATH, body=notif_payload, headers=notif_headers)
    notif_resp = conn.getresponse()
    notif_resp.read()  # consume response

    # Step 3: Call the tool
    tool_payload = json.dumps({
        "jsonrpc": "2.0",
        "method": "tools/call",
        "params": {
            "name": tool_name,
            "arguments": arguments
        },
        "id": 2
    })
    tool_headers = dict(headers)
    if session_id:
        tool_headers["Mcp-Session-Id"] = session_id

    conn.request("POST", MCP_PATH, body=tool_payload, headers=tool_headers)
    tool_resp = conn.getresponse()
    tool_body = tool_resp.read().decode("utf-8")
    tool_ct = tool_resp.getheader("Content-Type", "")

    conn.close()

    tool_result = parse_mcp_response(tool_body, tool_ct)

    if "error" in tool_result:
        return {"error": f"MCP tool call failed: {tool_result['error']}"}

    return tool_result.get("result", {})


# Quick smoke test of the helper
print("_call_you_mcp_tool helper defined \u2705")

In [ ]:
def you_web_search(
    query: str
) -> str:
    """
    Search the web using You.com free tier. Returns web results with titles,
    descriptions, and snippets. Use this tool to find real-time information,
    news, and general knowledge from the internet.

    Args:
        query (str): The search query. Supports search operators.
            Examples: "latest NVIDIA earnings", "site:reuters.com Tesla"

    Returns:
        str: JSON string with search results containing titles, URLs,
            descriptions, and snippets for each result.
    """
    import json
    import http.client

    if not query or not query.strip():
        return json.dumps({"error": "query cannot be empty"})

    MCP_HOST = "api.you.com"
    MCP_PATH = "/mcp"

    arguments = {"query": query.strip()}

    try:
        conn = http.client.HTTPSConnection(MCP_HOST, timeout=60)
        headers = {
            "Content-Type": "application/json",
            "Accept": "application/json, text/event-stream"
        }

        def parse_mcp_response(body, content_type):
            if "text/event-stream" in (content_type or ""):
                for line in body.split("\n"):
                    if line.startswith("data: "):
                        try:
                            msg = json.loads(line[6:])
                            if "result" in msg or "error" in msg:
                                return msg
                        except json.JSONDecodeError:
                            continue
                return {"error": {"message": "No valid response in SSE stream"}}
            else:
                return json.loads(body)

        # Initialize MCP session
        init_payload = json.dumps({
            "jsonrpc": "2.0",
            "method": "initialize",
            "params": {
                "protocolVersion": "2024-11-05",
                "capabilities": {},
                "clientInfo": {"name": "databricks-uc", "version": "1.0"}
            },
            "id": 1
        })
        conn.request("POST", MCP_PATH, body=init_payload, headers=headers)
        init_resp = conn.getresponse()
        init_body = init_resp.read().decode("utf-8")
        session_id = init_resp.getheader("Mcp-Session-Id") or init_resp.getheader("mcp-session-id")

        # Send initialized notification
        notif_headers = dict(headers)
        if session_id:
            notif_headers["Mcp-Session-Id"] = session_id
        conn.request("POST", MCP_PATH, body=json.dumps({
            "jsonrpc": "2.0",
            "method": "notifications/initialized"
        }), headers=notif_headers)
        conn.getresponse().read()

        # Call the search tool
        tool_headers = dict(headers)
        if session_id:
            tool_headers["Mcp-Session-Id"] = session_id
        tool_payload = json.dumps({
            "jsonrpc": "2.0",
            "method": "tools/call",
            "params": {"name": "you-search", "arguments": arguments},
            "id": 2
        })
        conn.request("POST", MCP_PATH, body=tool_payload, headers=tool_headers)
        tool_resp = conn.getresponse()
        tool_body = tool_resp.read().decode("utf-8")
        tool_ct = tool_resp.getheader("Content-Type", "")
        conn.close()

        result = parse_mcp_response(tool_body, tool_ct)

        if "error" in result:
            return json.dumps({"error": str(result["error"])})

        # Extract text content from MCP tool result
        content_parts = result.get("result", result).get("content", [])
        text_parts = [p.get("text", "") for p in content_parts if p.get("type") == "text"]
        return text_parts[0] if text_parts else json.dumps(result)

    except Exception as e:
        return json.dumps({"error": str(e)})


print("you_web_search function defined \u2705")

In [ ]:
def you_content_extract(
    url: str
) -> str:
    """
    Extract readable text content from a URL. Fetches the page directly via
    HTTP and strips HTML tags to return clean text. Use this to read and
    analyze web pages, articles, and documentation.

    Args:
        url (str): The URL to extract content from.
            Example: "https://example.com/article"

    Returns:
        str: The extracted page content as plain text (up to 10000 chars).
    """
    import json
    import http.client
    import re
    from urllib.parse import urlparse

    if not url or not url.strip():
        return json.dumps({"error": "url cannot be empty"})

    url = url.strip()

    try:
        parsed = urlparse(url)
        host = parsed.hostname
        path = parsed.path or "/"
        if parsed.query:
            path = path + "?" + parsed.query
        use_https = parsed.scheme != "http"

        if use_https:
            conn = http.client.HTTPSConnection(host, timeout=30)
        else:
            conn = http.client.HTTPConnection(host, timeout=30)

        req_headers = {
            "User-Agent": "Mozilla/5.0 (compatible; DatabricksUCFunction/1.0)",
            "Accept": "text/html,application/xhtml+xml,text/plain,*/*"
        }

        conn.request("GET", path, headers=req_headers)
        resp = conn.getresponse()

        # Follow redirects (up to 3)
        redirects = 0
        while resp.status in (301, 302, 303, 307, 308) and redirects < 3:
            location = resp.getheader("Location", "")
            resp.read()
            if not location:
                break
            if location.startswith("http"):
                parsed = urlparse(location)
                host = parsed.hostname
                path = parsed.path or "/"
                if parsed.query:
                    path = path + "?" + parsed.query
                use_https = parsed.scheme != "http"
                conn.close()
                if use_https:
                    conn = http.client.HTTPSConnection(host, timeout=30)
                else:
                    conn = http.client.HTTPConnection(host, timeout=30)
            else:
                path = location
            conn.request("GET", path, headers=req_headers)
            resp = conn.getresponse()
            redirects += 1

        if resp.status != 200:
            conn.close()
            return json.dumps({"error": f"HTTP {resp.status} {resp.reason}"})

        body = resp.read()
        conn.close()

        # Decode
        content_type = resp.getheader("Content-Type", "")
        charset = "utf-8"
        if "charset=" in content_type:
            charset = content_type.split("charset=")[-1].split(";")[0].strip()
        try:
            text = body.decode(charset)
        except (UnicodeDecodeError, LookupError):
            text = body.decode("utf-8", errors="replace")

        # If it looks like HTML, strip tags and extract text
        if "<html" in text.lower() or "<body" in text.lower() or "<!doctype" in text.lower():
            # Remove script and style blocks
            text = re.sub(r"<script[^>]*>.*?</script>", " ", text, flags=re.DOTALL | re.IGNORECASE)
            text = re.sub(r"<style[^>]*>.*?</style>", " ", text, flags=re.DOTALL | re.IGNORECASE)
            # Remove HTML comments
            text = re.sub(r"<!--.*?-->", " ", text, flags=re.DOTALL)
            # Convert <br>, <p>, <div>, headings to newlines
            text = re.sub(r"<(br|p|div|h[1-6]|li|tr)[^>]*>", "\n", text, flags=re.IGNORECASE)
            # Strip remaining tags
            text = re.sub(r"<[^>]+>", " ", text)
            # Decode HTML entities
            text = text.replace("&amp;", "&").replace("&lt;", "<").replace("&gt;", ">")
            text = text.replace("&quot;", '"').replace("&nbsp;", " ").replace("&#39;", "'")

        # Clean up whitespace
        text = re.sub(r"[ \t]+", " ", text)
        text = re.sub(r"\n\s*\n", "\n\n", text)
        text = text.strip()

        # Truncate to 10000 chars
        if len(text) > 10000:
            text = text[:10000] + "\n\n[Content truncated at 10000 characters]"

        return text if text else json.dumps({"error": "No text content found on page"})

    except Exception as e:
        return json.dumps({"error": str(e)})


print("you_content_extract function defined")

In [ ]:
def you_research(
    query: str
) -> str:
    """
    Perform web research on a topic by running a web search and extracting
    content from the top results. Returns a structured summary with sources,
    titles, URLs, and content snippets. Use this for in-depth analysis,
    comparisons, or when you need comprehensive answers with citations.

    Args:
        query (str): The research question or topic.
            Examples: "What are the latest trends in AI chip manufacturing?",
                     "Compare Tesla and BYD EV sales in 2024"

    Returns:
        str: JSON string with research results including source URLs,
            titles, and content snippets from top results.
    """
    import json
    import http.client
    import re
    from urllib.parse import urlparse

    if not query or not query.strip():
        return json.dumps({"error": "query cannot be empty"})

    MCP_HOST = "api.you.com"
    MCP_PATH = "/mcp"

    try:
        # Step 1: Search using You.com MCP (free tier)
        conn = http.client.HTTPSConnection(MCP_HOST, timeout=60)
        headers = {
            "Content-Type": "application/json",
            "Accept": "application/json, text/event-stream"
        }

        def parse_mcp_response(body, content_type):
            if "text/event-stream" in (content_type or ""):
                for line in body.split("\n"):
                    if line.startswith("data: "):
                        try:
                            msg = json.loads(line[6:])
                            if "result" in msg or "error" in msg:
                                return msg
                        except json.JSONDecodeError:
                            continue
                return {"error": {"message": "No valid response in SSE stream"}}
            else:
                return json.loads(body)

        # Initialize MCP session
        conn.request("POST", MCP_PATH, body=json.dumps({
            "jsonrpc": "2.0",
            "method": "initialize",
            "params": {
                "protocolVersion": "2024-11-05",
                "capabilities": {},
                "clientInfo": {"name": "databricks-uc", "version": "1.0"}
            },
            "id": 1
        }), headers=headers)
        init_resp = conn.getresponse()
        init_resp.read()
        session_id = init_resp.getheader("Mcp-Session-Id") or init_resp.getheader("mcp-session-id")

        # Send initialized notification
        notif_headers = dict(headers)
        if session_id:
            notif_headers["Mcp-Session-Id"] = session_id
        conn.request("POST", MCP_PATH, body=json.dumps({
            "jsonrpc": "2.0",
            "method": "notifications/initialized"
        }), headers=notif_headers)
        conn.getresponse().read()

        # Search
        tool_headers = dict(headers)
        if session_id:
            tool_headers["Mcp-Session-Id"] = session_id
        conn.request("POST", MCP_PATH, body=json.dumps({
            "jsonrpc": "2.0",
            "method": "tools/call",
            "params": {"name": "you-search", "arguments": {"query": query.strip(), "count": 5}},
            "id": 2
        }), headers=tool_headers)
        tool_resp = conn.getresponse()
        tool_body = tool_resp.read().decode("utf-8")
        tool_ct = tool_resp.getheader("Content-Type", "")
        conn.close()

        search_result = parse_mcp_response(tool_body, tool_ct)

        if "error" in search_result:
            return json.dumps({"error": str(search_result["error"])})

        # Parse search results to get URLs
        content_parts = search_result.get("result", search_result).get("content", [])
        search_text = ""
        for p in content_parts:
            if p.get("type") == "text":
                search_text = p.get("text", "")
                break

        # Try to parse search text as JSON to extract URLs
        urls_to_fetch = []
        try:
            search_data = json.loads(search_text)
            results = search_data.get("results", {})
            for source in ["web", "news"]:
                for item in results.get(source, [])[:3]:
                    urls_to_fetch.append({
                        "url": item.get("url", ""),
                        "title": item.get("title", "")
                    })
        except (json.JSONDecodeError, AttributeError):
            pass

        # Step 2: Fetch content from top URLs
        research_results = []
        for item in urls_to_fetch[:3]:
            fetch_url = item["url"]
            if not fetch_url:
                continue
            try:
                parsed = urlparse(fetch_url)
                host = parsed.hostname
                path = parsed.path or "/"
                if parsed.query:
                    path = path + "?" + parsed.query

                fc = http.client.HTTPSConnection(host, timeout=15)
                fc.request("GET", path, headers={
                    "User-Agent": "Mozilla/5.0 (compatible; DatabricksUCFunction/1.0)",
                    "Accept": "text/html,text/plain,*/*"
                })
                fresp = fc.getresponse()

                # Follow one redirect
                if fresp.status in (301, 302, 303, 307, 308):
                    loc = fresp.getheader("Location", "")
                    fresp.read()
                    if loc and loc.startswith("http"):
                        p2 = urlparse(loc)
                        fc.close()
                        fc = http.client.HTTPSConnection(p2.hostname, timeout=15)
                        rpath = p2.path or "/"
                        if p2.query:
                            rpath = rpath + "?" + p2.query
                        fc.request("GET", rpath, headers={
                            "User-Agent": "Mozilla/5.0 (compatible; DatabricksUCFunction/1.0)",
                            "Accept": "text/html,text/plain,*/*"
                        })
                        fresp = fc.getresponse()

                if fresp.status == 200:
                    fbody = fresp.read()
                    try:
                        ftext = fbody.decode("utf-8")
                    except UnicodeDecodeError:
                        ftext = fbody.decode("utf-8", errors="replace")

                    # Strip HTML
                    ftext = re.sub(r"<script[^>]*>.*?</script>", " ", ftext, flags=re.DOTALL | re.IGNORECASE)
                    ftext = re.sub(r"<style[^>]*>.*?</style>", " ", ftext, flags=re.DOTALL | re.IGNORECASE)
                    ftext = re.sub(r"<[^>]+>", " ", ftext)
                    ftext = re.sub(r"[ \t]+", " ", ftext)
                    ftext = re.sub(r"\n\s*\n", "\n", ftext).strip()

                    # Take first 2000 chars as snippet
                    snippet = ftext[:2000] if len(ftext) > 2000 else ftext

                    research_results.append({
                        "url": fetch_url,
                        "title": item["title"],
                        "snippet": snippet
                    })

                fc.close()
            except Exception:
                research_results.append({
                    "url": fetch_url,
                    "title": item["title"],
                    "snippet": "[Could not fetch content]"
                })

        output = {
            "query": query.strip(),
            "source_count": len(research_results),
            "search_summary": search_text[:2000] if search_text else "",
            "sources": research_results
        }

        return json.dumps(output, ensure_ascii=False)

    except Exception as e:
        return json.dumps({"error": str(e)})


print("you_research function defined")

## Step 3: Register as UC Functions

In [ ]:
from unitycatalog.ai.core.databricks import DatabricksFunctionClient
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
client = DatabricksFunctionClient(client=w)

functions = [
    ("you_web_search", you_web_search),
    ("you_content_extract", you_content_extract),
    ("you_research", you_research),
]

for func_name, func_obj in functions:
    full_name = f"{catalog}.{schema}.{func_name}"
    print(f"Creating UC Function: {full_name}...")
    function_info = client.create_python_function(
        func=func_obj,
        catalog=catalog,
        schema=schema,
        replace=True
    )
    print(f"Created: {function_info.full_name}")

print(f"\nAll 3 UC Functions registered in {catalog}.{schema}")
print(f"MCP URL: {w.config.host}/api/2.0/mcp/functions/{catalog}/{schema}")

## Step 4: Test the UC Functions

In [ ]:
import json

# Test 1: Web Search
print("Test 1: Web Search")
print("-" * 60)
result = client.execute_function(
    f"{catalog}.{schema}.you_web_search",
    {"query": "latest NVIDIA earnings 2025"}
)
output = result.value
print(f"\u2705 Result preview: {output[:500]}..." if len(output) > 500 else f"\u2705 Result: {output}")
print()

In [ ]:
# Test 2: Content Extraction
print("Test 2: Content Extraction")
print("-" * 60)
result = client.execute_function(
    f"{catalog}.{schema}.you_content_extract",
    {"url": "https://docs.databricks.com/en/getting-started/index.html"}
)
output = result.value
print(f"\u2705 Result preview: {output[:500]}..." if len(output) > 500 else f"\u2705 Result: {output}")
print()

In [ ]:
# Test 3: Research
print("Test 3: Research")
print("-" * 60)
result = client.execute_function(
    f"{catalog}.{schema}.you_research",
    {"query": "Databricks vs Snowflake comparison 2025"}
)
output = result.value
print(f"\u2705 Result preview: {output[:500]}..." if len(output) > 500 else f"\u2705 Result: {output}")

## Done! Web Search UC Functions Are Ready

### What You Created:
- `you_web_search` -- Real-time web and news search (You.com MCP free tier)
- `you_content_extract` -- Direct URL content extraction (HTTP fetch, no API needed)
- `you_research` -- Search + extract from top results (combines both above)

### Next Step:
Run `04_instructor_setup_sa.ipynb` to create the Supervisor Agent -- it will automatically discover and include these functions.

### How It Works:
```
User Question -> Supervisor Agent -> you_web_search (You.com MCP free tier)
                                  -> you_content_extract (direct HTTP fetch)
                                  -> you_research (search + fetch top results)
```

### Example Queries for the Supervisor:
- "Search the web for the latest NVIDIA earnings report"
- "Read the content of this article: https://example.com/article"
- "Research the current state of AI chip competition between NVIDIA and AMD"